In [2]:
import tensorflow as tf

print(tf.__version__)


2.15.0


In [3]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [4]:
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [11]:
# 이게 핵심

class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name   # 게이트 이름
        self.xdata = xdata.reshape(4,2)   # 학습 데이터
        self.tdata = tdata.reshape(4,1)   # 정답 데이터

        self.W = np.random.rand(self.xdata.shape[1], 1)   # 가중치 초기값 설정
        self.b = np.random.rand(1)   # 바이어스 초기값 설정
        self.learning_rate = 1e-2    # 학습율

    def loss_func(self):
        delta = 1e-7
        z = np.dot(self.xdata, self.W) + self.b
        y = sigmoid(z)   # 활성함수: 시그모이드 함수 = 이진 분류에서 사용함
        return -np.sum(self.tdata*np.log(y+delta) + (1-self.tdata)*np.log((1-y)+delta))   # 이진 분류에 대한 손실값 / 이진 분류일 때 이 수식을 사용함

    
# 학습하는 과정을 함수로 만들 거임
    def train(self):
        f = lambda x : self.loss_func()   # 손실값 계산하는 람다 함수
        print("Initial loss value = ", self.loss_func())   # 초기 손실값 계산해서 출력
        for step in range(8001):
            self.W -= self.learning_rate * numerical_derivative(f, self.W)   # 가중치 업데이터
            self.b -= self.learning_rate * numerical_derivative(f, self.b)   # 바이어스 업데이트
            if (step % 1000 == 0):
                print("step = ", step, "loss value = ", self.loss_func())    # 헉습중 손실값 출력


# 예측하는 거 만들 거임
    def predict(self, input_data):
        z = np.dot(input_data, self.W) + self.b   # 회귀식 계산
        y = sigmoid(z)   # 활성 함수 시그모이드 함수 실행: 0~1 사이의 확률값 출력
        if y > 0.5:
            result = 1   # 분류값 저장
        else:
            result = 0
        return y, result

    
# 정확도 계산하는 거 만들 거임
    def accuracy(self, test_xdata, test_tdata):
        matched_list = []   # 정답과 일치한 내용 저장
        not_matched_list = []   # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)   # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)   # 정답과 불일치한 내용 추가
        accuracy_val = len(matched_list) / len(test_xdata)   # 정확도 계산
        return accuracy_val

In [12]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
tdata = np.array([0, 0, 0, 1])
AND_obj = LogicGate("AND_GATE", xdata, tdata)
AND_obj.train()

Initial loss value =  4.683137495776714
step =  0 loss value =  4.623211601768596
step =  1000 loss value =  1.0069465426501885
step =  2000 loss value =  0.6602310546524506
step =  3000 loss value =  0.4913958191713578
step =  4000 loss value =  0.3902625398826987
step =  5000 loss value =  0.32289948283960956
step =  6000 loss value =  0.2748993687964382
step =  7000 loss value =  0.23902934021950262
step =  8000 loss value =  0.2112500974695601


In [13]:
test_xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = AND_obj.predict(input_data)
    print("입력값 = ", logical_val)
    
    
print("---------------------------------")
test_tdata = np.array([0, 0, 0, 1])
accuracy_val = AND_obj.accuracy(test_xdata, test_tdata)
print("정확도 = ", accuracy_val)

입력값 =  0
입력값 =  0
입력값 =  0
입력값 =  1
---------------------------------
정확도 =  1.0


In [14]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
tdata = np.array([0, 1, 1, 1])
OR_obj = LogicGate("OR_GATE", xdata, tdata)
OR_obj.train()

Initial loss value =  1.8100970690820732
step =  0 loss value =  1.8055816158257105
step =  1000 loss value =  0.6992251036799759
step =  2000 loss value =  0.4236096440005283
step =  3000 loss value =  0.29951818157094134
step =  4000 loss value =  0.23008500560645606
step =  5000 loss value =  0.18611119623432437
step =  6000 loss value =  0.15591945935056847
step =  7000 loss value =  0.1339781095129511
step =  8000 loss value =  0.11734638677189954


In [15]:
test_xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = OR_obj.predict(input_data)
    print(input_data," = ", logical_val)
    
    
print("---------------------------------")
test_tdata = np.array([0, 1, 1, 1])
accuracy_val = OR_obj.accuracy(test_xdata, test_tdata)
print("정확도 = ", accuracy_val)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  1
---------------------------------
정확도 =  1.0


In [ ]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
tdata = np.array([1, 1, 1, 0])
NAND_obj = LogicGate("NAND_GATE", xdata, tdata)
NAND_obj.train()

Initial loss value =  3.116625833339949
step =  0 loss value =  3.1049865294403407
step =  1000 loss value =  1.0559720036428084
step =  2000 loss value =  0.6803670262421246
step =  3000 loss value =  0.5025784861282807
step =  4000 loss value =  0.3973856796181098
step =  5000 loss value =  0.3278236818962103
step =  6000 loss value =  0.27849849518753655
step =  7000 loss value =  0.24176961103557423
step =  8000 loss value =  0.21340294738624505


In [17]:
test_xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = NAND_obj.predict(input_data)
    print(input_data," = ", logical_val)
    
    
print("---------------------------------")
test_tdata = np.array([1, 1, 1, 0])
accuracy_val = NAND_obj.accuracy(test_xdata, test_tdata)
print("정확도 = ", accuracy_val)

[0 0]  =  1
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0
---------------------------------
정확도 =  1.0


In [18]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
tdata = np.array([0, 1, 1, 0])
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train()

Initial loss value =  3.046399290563183
step =  0 loss value =  3.038832013370083
step =  1000 loss value =  2.7730216808287707
step =  2000 loss value =  2.772606413871054
step =  3000 loss value =  2.7725887126199957
step =  4000 loss value =  2.772587956037401
step =  5000 loss value =  2.772587923685182
step =  6000 loss value =  2.7725879223016694
step =  7000 loss value =  2.7725879222425043
step =  8000 loss value =  2.772587922239974


In [19]:
test_xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = XOR_obj.predict(input_data)
    print(input_data," = ", logical_val)
    
    
print("---------------------------------")
test_tdata = np.array([0, 1, 1, 0])
accuracy_val = XOR_obj.accuracy(test_xdata, test_tdata)
print("정확도 = ", accuracy_val)

[0 0]  =  0
[0 1]  =  0
[1 0]  =  0
[1 1]  =  1
---------------------------------
정확도 =  0.25


In [23]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]])
tdata = np.array([0, 1, 1, 0])

AND_tdata = np.array([0, 0, 0, 1])
OR_tdata = np.array([0, 1, 1, 1])
NAND_tdata = np.array([1, 1, 1, 0])

AND_obj = LogicGate("AND_GATE", xdata, AND_tdata)
OR_obj = LogicGate("OR_GATE", xdata, OR_tdata)
NAND_obj = LogicGate("NAND_GATE", xdata, NAND_tdata)

AND_obj.train()
OR_obj.train()
NAND_obj.train()

Initial loss value =  3.3546813122516963
step =  0 loss value =  3.3258373936246914
step =  1000 loss value =  1.0290808408577266
step =  2000 loss value =  0.6694428748037566
step =  3000 loss value =  0.4965346933006024
step =  4000 loss value =  0.39354373632012896
step =  5000 loss value =  0.3251712641746549
step =  6000 loss value =  0.2765616493449895
step =  7000 loss value =  0.24029599771718665
step =  8000 loss value =  0.21224586948648877
Initial loss value =  1.9813567560540915
step =  0 loss value =  1.9763455640514263
step =  1000 loss value =  0.7350627030541625
step =  2000 loss value =  0.43770966732377736
step =  3000 loss value =  0.30681606537515455
step =  4000 loss value =  0.23447927032472954
step =  5000 loss value =  0.1890240160229823
step =  6000 loss value =  0.157982476867528
step =  7000 loss value =  0.13551150080290836
step =  8000 loss value =  0.11852867588503464
Initial loss value =  2.7785740242437873
step =  0 loss value =  2.773058678149296
step =

In [24]:
test_xdata = np.array([[0,0], [0,1], [1,0], [1,1]])

for input_data in test_xdata:
    (sigmoid_val, s1) = NAND_obj.predict(input_data)
    (sigmoid_val, s2) = OR_obj.predict(input_data)
    new_input_data = np.array([s1, s2])
    (sigmoid_val, logical_val) = AND_obj.predict(new_input_data)

    print(input_data, " = ", logical_val)
    
    
print("---------------------------------")
test_tdata = np.array([0, 1, 1, 0])
accuracy_val = AND_obj.accuracy(test_xdata, test_tdata)
print("정확도 = ", accuracy_val)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0
---------------------------------
정확도 =  0.25


In [25]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
a = []
b = []
for input_data in test_xdata:
    (sigmoid_val1, logical_val1) = NAND_obj.predict(input_data)
    (sigmoid_val2, logical_val2) = OR_obj.predict(input_data)
    a.append(logical_val1)
    b.append(logical_val2)

lst = []
i = 0
while i < len(a):
    lst.append([a[i], b[i]])
    i += 1
print(lst)
print('---------------------')

for input_data in lst:
    (sigmoid_val, logical_val) = AND_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('---------------------')
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = AND_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[[1, 0], [1, 1], [1, 1], [0, 1]]
---------------------
[1, 0] = 0
[1, 1] = 1
[1, 1] = 1
[0, 1] = 0
---------------------
Accuracy => 0.25


In [26]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = XOR_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------')
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = XOR_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
----------------
Accuracy => 0.25
